In [2]:
import os
import mlflow

TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "https://mlflow-dashboard.duckdns.org"
)

mlflow.set_tracking_uri(TRACKING_URI)

with mlflow.start_run():
    mlflow.log_param("param1", 15)
    mlflow.log_metric("metric1", 0.89)

🏃 View run beautiful-bat-873 at: https://mlflow-dashboard.duckdns.org/#/experiments/0/runs/82574b6f85614debb29f80eba99a4b6b
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/0


## Perform the Same EDA

In [3]:
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [4]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

class TextCleaner:
    """Class to clean and preprocess raw text comments."""
    def __init__(self):
        self._ensure_nltk_resources()
        self.stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
        self.lemmatizer = WordNetLemmatizer()
        
    def _ensure_nltk_resources(self):
        try:
            nltk.data.find('corpora/stopwords')
        except LookupError:
            nltk.download('stopwords', quiet=True)
        try:
            nltk.data.find('corpora/wordnet')
        except LookupError:
            nltk.download('wordnet', quiet=True)
            
    def clean_text(self, text):
        if not isinstance(text, str):
            return ""
            
        # Convert to lowercase
        text = text.lower()
        
        # Remove trailing and leading whitespaces
        text = text.strip()
        
        # Remove newline characters
        text = re.sub(r'\n', ' ', text)
        
        # Remove URLs
        url_pattern = r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+"
        text = re.sub(url_pattern, '', text)
        
        # Remove non-alphanumeric characters, except punctuation
        text = re.sub(r'[^A-Za-z0-9\s!?.,]', '', text)
        
        # Remove stopwords
        text = ' '.join(word for word in text.split() if word not in self.stop_words)
        
        # Lemmatize words
        text = ' '.join(self.lemmatizer.lemmatize(word) for word in text.split())
        
        return text

class DataPreprocessor:
    """Class to handle DataFrame level data preprocessing."""
    def __init__(self, text_column='clean_comment'):
        self.text_column = text_column
        self.text_cleaner = TextCleaner()
        
    def fit(self, df, y=None):
        return self
        
    def transform(self, df):
        """Executes the full preprocessing pipeline on the input DataFrame."""
        df_processed = df.copy()
        
        if self.text_column not in df_processed.columns:
            return df_processed
            
        # 1. Drop missing values based on text column
        df_processed.dropna(subset=[self.text_column], inplace=True)
        
        # 2. Drop duplicates
        df_processed.drop_duplicates(inplace=True)
        
        # 3. Strip initial empty records before further processing
        df_processed = df_processed[df_processed[self.text_column].str.strip() != ""]
        
        # 4. Apply text cleaning
        df_processed[self.text_column] = df_processed[self.text_column].apply(self.text_cleaner.clean_text)
        
        # 5. Remove any empty strings formed after cleaning
        df_processed = df_processed[df_processed[self.text_column].str.strip() != ""]
        
        return df_processed


In [5]:
import sys
import os

# Add the parent directory so Python can find the 'src' module
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.data_preprocessing import DataPreprocessor

# 1. Load your raw data
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')

# 2. Initialize the preprocessor and transform the data
preprocessor = DataPreprocessor(text_column='clean_comment')
df_clean = preprocessor.transform(df)

# Check the fully cleaned results
df_clean.head()


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [6]:
df_clean.shape

(36662, 2)

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns